# Econ 5200 PS2 — The Underpowered Portfolio

**ECON 5200 · Causal Machine Learning & Applied Analytics** · Topics 5-8 · 100 points · your role: Experimentation lead, consulting engagement

This is the **starter**: the structure of the set in rubric order. Fill the empty cells; keep every heading so the grader finds each step. The brief on Canvas has the full text of every step, the numbers you must hit and the rubric. Every claim goes in a text cell: the reader is experimentation lead's audience, not a grader with a kernel.

**AI is not allowed in Phases 1–3.** Generative AI is **prohibited** in Phases 1 through 3. AI is authorised and required in Phase 4. You must be able to defend every line under questioning.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw reruns the same way
pd.set_option('display.precision', 3)

import os
os.makedirs('src', exist_ok=True)   # the .py module the brief asks for goes here

## Phase 1: Reconstruct the Portfolio (30 pts)

### Step 1.1: Simulate the experiment programme (15 pts)

In [ ]:
rng = np.random.default_rng(3)
n_experiments = 200

share = 0.25
outcome_sd = 1.0

real = rng.random(n_experiments) < share

truth = np.where(
    real,
    rng.normal(0.30, 0.10, size=n_experiments),
    0.0
)

sample_size = rng.integers(40, 400, size=n_experiments)
standard_error = outcome_sd * np.sqrt(2 / sample_size)

effect_estimate = truth + rng.normal(
    0, standard_error, size=n_experiments
)

experiments = pd.DataFrame({
    "Truth": truth,
    "Sample_Size": sample_size,
    "Standard_Error": standard_error,
    "Effect_Estimate": effect_estimate
})

print("Number of experiments:", len(experiments))
print("Mean true effect:", experiments["Truth"].mean())
print("Mean estimated effect:", experiments["Effect_Estimate"].mean())

Number of experiments: 200
Mean true effect: 0.06962106991581968
Mean estimated effect: 0.08546291801527929


*Your answer here.*

I assume that 25% of experiments have a real effect drawn from a normal distribution while 75% have no true effect. I recore each experiments true effect in the Truth column before generating an observed estimate. I then add rendom sampling noise to each true effect, with the amount of noise determined by the standard error and sample size.

### Step 1.2: Apply the client's rule (15 pts)

> Add blockquote



In [ ]:
experiments["z_score"] = (
    experiments["Effect_Estimate"] / experiments["Standard_Error"]
)

experiments["p_value"] = 2 * stats.norm.sf(
    np.abs(experiments["z_score"])
)

experiments["Ship"] = experiments["p_value"] < 0.05

shipped = experiments[experiments["Ship"]]

print("Experiments shipped:", len(shipped))
print("Mean estimated effect:", round(shipped["Effect_Estimate"].mean(), 3))
print("Mean true effect:", round(shipped["Truth"].mean(), 3))
print(
    "False-ship rate:",
    round((shipped["Truth"] == 0).mean() * 100, 1), "%"
)

Experiments shipped: 44
Mean estimated effect: 0.307
Mean true effect: 0.275
False-ship rate: 18.2 %


*Your answer here.*

This adds the z score for each experiment by divding the effect estimate by the standard error. This is then used to calculate the p value to test weather the true effect is 0. The shipping rule which states that an experiment is shiped if it has a p value under 0.05. The code then calculates the number and fraction of experiments shipped, along with the mean true effect and mean estimated effect across all 200 experiments.

## Phase 2: The Winner's Curse (35 pts)

### Step 2.1: Quantify the exaggeration (20 pts)

In [ ]:

shipped = experiments[experiments["p_value"] < 0.05].copy()

mean_estimated_effect = shipped["Effect_Estimate"].mean()
mean_true_effect = shipped["Truth"].mean()

absolute_inflation = mean_estimated_effect - mean_true_effect

relative_inflation = absolute_inflation / mean_true_effect

print("Shipped:", len(shipped))
print("Mean estimated effect:", round(mean_estimated_effect, 4))
print("Mean true effect:", round(mean_true_effect, 4))
print("Inflation:", round(absolute_inflation, 4))

z_critical = stats.norm.ppf(0.975)
effect_to_se = experiments["Truth"] / experiments["Standard_Error"]

experiments["Power"] = (
    stats.norm.cdf(-z_critical - effect_to_se)
    + stats.norm.sf(z_critical - effect_to_se)
)

experiments["Power_Tier"] = pd.cut(
    experiments["Power"],
    bins=[-np.inf, 0.40, 0.80, np.inf],
    labels=["Low", "Medium", "High"]
)

shipped = experiments[experiments["p_value"] < 0.05].copy()

tier_results = shipped.groupby(
    "Power_Tier", observed=True
).agg(
    Number_Shipped=("Truth", "size"),
    Mean_True_Effect=("Truth", "mean"),
    Mean_Estimated_Effect=("Effect_Estimate", "mean")
)

tier_results["Absolute_Inflation"] = (
    tier_results["Mean_Estimated_Effect"]
    - tier_results["Mean_True_Effect"]
)


print("\nInflation by power tier:")
print(tier_results.round(4))

Shipped: 44
Mean estimated effect: 0.3066
Mean true effect: 0.2754
Inflation: 0.0312

Inflation by power tier:
            Number_Shipped  Mean_True_Effect  Mean_Estimated_Effect  \
Power_Tier                                                            
Low                     10             0.033                  0.023   
Medium                   6             0.317                  0.376   
High                    28             0.353                  0.393   

            Absolute_Inflation  
Power_Tier                      
Low                     -0.010  
Medium                   0.059  
High                     0.040  


*Your answer here.*

I first compare the average stimated effect of shipped experiments with their average true effect to measure inflation. Then I calculate the power of each experiment and group them into low, medium or higher power tiers. Finally, I calculate the false ship rate whish is teh percentage of shiped experments whose true effect is 0.


### Step 2.2: The bootstrap where the formula fails (15 pts)

In [ ]:
realized_effects = shipped["Truth"].to_numpy()


rn = np.random.default_rng(42)
n_bootstrap = 10000
bootstrap_mean = []

for _ in range(n_bootstrap):
  sample = rn.choice(
      realized_effects,
      size=len(realized_effects),
      replace=True
  )
  bootstrap_mean.append(sample.mean())

bootstrap_mean = np.array(bootstrap_mean)

print("Realized return:", round(realized_effects.mean(), 4))
print(
    "Manual bootstrap 95% interval:",
    np.round(np.percentile(bootstrap_mean, [2.5, 97.5]), 4)
)

result = stats.bootstrap(
    (realized_effects,),
    np.mean,
    confidence_level=0.95,
    n_resamples=10000,
    method="BCa",
    rng=np.random.default_rng(42)
)

print("SciPy BCa 95% interval:",
      np.round([result.confidence_interval.low,
                result.confidence_interval.high], 4))

Realized return: 0.2754
Manual bootstrap 95% interval: [0.2286 0.3216]
SciPy BCa 95% interval: [0.2271 0.3197]


*Your answer here.*

As you can see above they are both very similar. They vary by about 0.01. This may be because my manual version takes the 2.5th and 97.5th percentile of 10,000 sample means, while the other method uses BCa to acount for bias in the bootstrap distribution which my function doesnt do.

## Phase 3: A Rule They Can Adopt (25 pts)

### Step 3.1: Design the alternative (20 pts)

In [ ]:
#Clients rule, ship if p < 0.05
client_rule = experiments["p_value"] < 0.05

#My rule, ship if estimated Cohen's d > 0.2
outcome_sd = 1.0
experiments["Cohens_d"] = (
    experiments["Effect_Estimate"] / outcome_sd
)

my_rule = experiments["Cohens_d"] > 0.2

def evaluate_rule(rule, experiments):
    selected = experiments[rule]

    return {
        "Realized Return": selected["Truth"].mean(),
        "Number Shipped": len(selected),
        "False-Ship Rate": (
            (selected["Truth"] == 0).mean()
            if len(selected) > 0 else 0
        )
    }

comparison = pd.DataFrame({
    "My Rule (Cohen's d)": evaluate_rule(my_rule, experiments),
    "Client's Rule (p < 0.05)": evaluate_rule(client_rule, experiments)
})

print(comparison.round(4))



                 My Rule (Cohen's d)  Client's Rule (p < 0.05)
Realized Return                0.269                     0.275
Number Shipped                48.000                    44.000
False-Ship Rate                0.188                     0.182


*Your answer here.*

The rule I propose is to use Cohen's d to ship experiments that meet a minimum effect size. Since I assumed the standard deviation is 1, Cohen's d is just the estimated effect divided by the standard deviation. This didn't work as well as the client's p-value rule because we shipped more experiments, had a higher false-ship rate, and had a lower realized return. This probably means my threshold wasn't high enough to pick out the experiments with meaningful effects.

### Step 3.2: Ship the module (5 pts, its own rubric row)

The module the brief asks for. Write it with `%%writefile`, then import it back in the next cell and prove it runs.

In [ ]:
from pathlib import Path
import os
Path("src").mkdir(parents=True, exist_ok=True)
print("Current folder:", os.getcwd())
print("Files here:", os.listdir("."))
print("Files in src:", os.listdir("src"))


Current folder: /content
Files here: ['.config', 'src', 'ai-appendix.md', '.ipynb_checkpoints', 'sample_data']
Files in src: ['__pycache__', 'experiment_portfolio.py']


In [ ]:
%%writefile src/experiment_portfolio.py
"""Experiment-portfolio simulator and decision rules (ECON 5200 PS2)."""
from __future__ import annotations
import numpy as np
import pandas as pd
from scipy import stats


def simulate_portfolio(n: int, seed: int, **kwargs):
    """Return the experiment log with the true effect attached."""
    rn = np.random.default_rng(seed)

    share = kwargs.get("share", 0.25)
    effect_mean = kwargs.get("effect_mean", 0.30)
    effect_sd = kwargs.get("effect_sd", 0.10)
    sample_size_low = kwargs.get("sample_size_low", 40)
    sample_size_high = kwargs.get("sample_size_high", 400)
    outcome_sd = kwargs.get("outcome_sd", 1.0)

    real = rn.random(n) < share

    truth = np.where(
        real,
        rn.normal(effect_mean, effect_sd, size=n),
        0.0
    )

    sample_size = rn.integers(
        sample_size_low, sample_size_high + 1, size=n
    )
    standard_error = outcome_sd * np.sqrt(2 / sample_size)

    estimate = truth + rn.normal(0, standard_error)

    log = pd.DataFrame({
        "Truth": truth,
        "Sample_Size": sample_size,
        "Outcome_SD": outcome_sd,
        "Standard_Error": standard_error,
        "Effect_Estimate": estimate
    })

    z_score = log["Effect_Estimate"] / log["Standard_Error"]
    log["p_value"] = 2 * stats.norm.sf(np.abs(z_score))
    log["Cohens_d"] = log["Effect_Estimate"] / outcome_sd

    alpha = 0.05
    minimum_effect = 0.30
    z_critical = stats.norm.ppf(1 - alpha / 2)

    noncentrality = minimum_effect / log["Standard_Error"]

    log["Power"] = (
        stats.norm.cdf(-z_critical - noncentrality)
        + stats.norm.sf(z_critical - noncentrality)
    )

    return log


def apply_rule(log, rule):
    """Ship decisions under `rule`: the p < 0.05 threshold and at least one other."""
    if rule == "p":
        return log["p_value"] < 0.05

    if rule == "cohens_d":
        return log["Cohens_d"] > 0.2

    if rule == "power":
        return (log["p_value"] < 0.05) & (log["Power"] >= 0.80)

    raise ValueError(
        "Unknown rule. Use 'p' or 'cohens_d'."
    )


def evaluate(log, rule):
    """Realised true return, ship count and false-ship rate under `rule`."""
    shipped = log[apply_rule(log, rule)]
    ship_count = len(shipped)

    if ship_count == 0:
        return {
            "Realized Return": 0.0,
            "Number Shipped": 0,
            "False-Ship Rate": np.nan,
            "Average True Effect": np.nan,
            "Average Estimated Effect": np.nan
        }

    return {
        "Realized Return": shipped["Truth"].sum(),
        "Number Shipped": ship_count,
        "False-Ship Rate": (shipped["Truth"] == 0).mean(),
        "Average True Effect": shipped["Truth"].mean(),
        "Average Estimated Effect": shipped["Effect_Estimate"].mean()
    }



if __name__ == "__main__":
    log = simulate_portfolio(200, seed=3)

    print("Client rule:", evaluate(log, "p"))
    print("Cohen's d rule:", evaluate(log, "cohens_d"))

    same_log = simulate_portfolio(200, seed=3)
    assert log.equals(same_log), "Same seed should reproduce the results."

    client_ships = apply_rule(log, "p")
    assert (client_ships == (log["p_value"] < 0.05)).all()

    d_ships = apply_rule(log, "cohens_d")
    assert (d_ships == (log["Cohens_d"] > 0.2)).all()

    assert 0 <= client_ships.sum() <= 200

    print("All assertions passed.")

Overwriting src/experiment_portfolio.py


In [ ]:
import importlib
from src import experiment_portfolio as ep

importlib.reload(ep)

log = ep.simulate_portfolio(200, seed=3)

print("Client rule:", ep.evaluate(log, "p"))
print("Cohen's d rule:", ep.evaluate(log, "cohens_d"))

small_log = ep.simulate_portfolio(10, seed=1)
ships = ep.apply_rule(small_log, "p")
results = ep.evaluate(small_log, "p")

assert len(small_log) == 10
assert len(ships) == 10
assert isinstance(results, dict)

print("All three functions run successfully.")

Client rule: {'Realized Return': np.float64(12.119199279650719), 'Number Shipped': 44, 'False-Ship Rate': np.float64(0.18181818181818182), 'Average True Effect': np.float64(0.27543634726478905), 'Average Estimated Effect': np.float64(0.30651996236794615)}
Cohen's d rule: {'Realized Return': np.float64(12.906862118677681), 'Number Shipped': 48, 'False-Ship Rate': np.float64(0.1875), 'Average True Effect': np.float64(0.268892960805785), 'Average Estimated Effect': np.float64(0.35700893544048645)}
All three functions run successfully.


## Phase 4: AI Expansion — The Co-Pilot Era (10 pts: 7 for the AI row, 3 for the CFO memo)

**Status: AI usage AUTHORISED and REQUIRED.** Use the **P.R.I.M.E. framework**.

### Task 4.1: The sequential design

In [ ]:
"""Always-valid sequential test: two-sided mixture SPRT (Gaussian mixing prior).
numpy + scipy only.

Stream model: each observation is a paired difference d_i = treatment_i - control_i,
d_i ~ N(theta, sigma_d^2) with sigma_d known. H0: theta = 0.
Mixing prior on theta: N(0, tau^2).

  Lambda_n = sqrt(s2/(s2 + n tau^2)) * exp( n^2 tau^2 dbar^2 / (2 s2 (s2 + n tau^2)) )

Under H0, Lambda_n is a nonnegative martingale with mean 1, so by Ville's inequality
P(sup_n Lambda_n >= 1/alpha) <= alpha.  Reject at the first n with Lambda_n >= 1/alpha.
"""
import numpy as np


def msprt(stream, alpha=0.05, sigma_d=1.0, tau=None):
    """Return the per-look log (columns as arrays, one entry per look).

    stream  : 1-D array of observations (paired differences), or 2-D (n, 2) array
              of (control, treatment) which is converted to differences.
    sigma_d : known SD of one observation (the difference).
    tau     : SD of the mixing prior on the effect; default 0.5 * sigma_d.
    """
    x = np.asarray(stream, dtype=float)
    if x.ndim == 2:
        x = x[:, 1] - x[:, 0]
    n = np.arange(1, x.size + 1, dtype=float)
    s2 = sigma_d ** 2
    tau = 0.5 * sigma_d if tau is None else tau
    t2 = tau ** 2
    mean = np.cumsum(x) / n
    log_lam = (0.5 * np.log(s2 / (s2 + n * t2))
               + (n ** 2 * t2 * mean ** 2) / (2 * s2 * (s2 + n * t2)))
    run_max = np.maximum.accumulate(log_lam)
    p_av = np.minimum(1.0, np.exp(-run_max))          # always-valid p-value
    reject = run_max >= np.log(1.0 / alpha)           # monotone: stays True once crossed
    return {"look": n.astype(int), "mean_diff": mean, "log_lambda": log_lam,
            "p_always_valid": p_av, "stop": reject}


def first_stop(log):
    idx = np.flatnonzero(log["stop"])
    return int(idx[0]) + 1 if idx.size else None


def naive_fixed_z(stream, alpha=0.05, sigma_d=1.0):
    """The wrong way: fixed-horizon z-test applied at every look."""
    from scipy.stats import norm
    x = np.asarray(stream, float)
    n = np.arange(1, x.size + 1)
    z = np.cumsum(x) / (sigma_d * np.sqrt(n))
    return np.abs(z) > norm.ppf(1 - alpha / 2)


if __name__ == "__main__":
    SEED, N_STREAMS, N_LOOKS, ALPHA, SIGMA = 20260427, 1000, 5000, 0.05, 1.0
    rng = np.random.default_rng(SEED)

    # --- pure null ---
    fp_m = fp_n = 0
    for _ in range(N_STREAMS):
        s = rng.normal(0.0, SIGMA, N_LOOKS)
        fp_m += msprt(s, ALPHA, SIGMA)["stop"][-1]
        fp_n += naive_fixed_z(s, ALPHA, SIGMA).any()
    print(f"[null] nominal alpha            : {ALPHA:.3f}")
    print(f"[null] mSPRT false positive rate: {fp_m/N_STREAMS:.3f}  ({fp_m}/{N_STREAMS})")
    print(f"[null] naive z-every-look FPR   : {fp_n/N_STREAMS:.3f}  ({fp_n}/{N_STREAMS})")
    se = np.sqrt(ALPHA * (1 - ALPHA) / N_STREAMS)
    print(f"[null] MC std. error at 5%      : {se:.4f}")

    # --- portfolio of 200 tests with known effects ---
    rng = np.random.default_rng(SEED + 1)
    eff = np.where(rng.random(200) < 0.5, 0.0, rng.choice([0.03, 0.05, 0.08, 0.12], 200))
    res = []
    for th in eff:
        s = rng.normal(th, SIGMA, N_LOOKS)
        res.append(first_stop(msprt(s, ALPHA, SIGMA)))
    res = np.array([r if r else 0 for r in res]); rej = res > 0
    null = eff == 0
    print(f"[portfolio] {null.sum()} nulls, {(~null).sum()} true effects")
    print(f"[portfolio] false positives: {rej[null].sum()}/{null.sum()}")
    print(f"[portfolio] power          : {rej[~null].mean():.3f}")
    print(f"[portfolio] median stop look among detected effects: {int(np.median(res[rej & ~null]))}")

[null] nominal alpha            : 0.050
[null] mSPRT false positive rate: 0.040  (40/1000)
[null] naive z-every-look FPR   : 0.620  (620/1000)
[null] MC std. error at 5%      : 0.0069
[portfolio] 93 nulls, 107 true effects
[portfolio] false positives: 3/93
[portfolio] power          : 0.710
[portfolio] median stop look among detected effects: 1022


In [ ]:

import numpy as np

N = 1000
T = 5000
alpha = 0.05
rng = np.random.default_rng(7)

msprt_rejections = 0
naive_rejections = 0

for _ in range(N):
    # Pure-null data: both groups have the same true mean
    control = rng.normal(0, 1, T)
    treatment = rng.normal(0, 1, T)

    # Always-valid sequential test
    result = msprt_stream(
        control, treatment,
        alpha=alpha,
        tau2=0.25,
        sigma2=1.0
    )
    msprt_rejections += result["stop_at"] is not None

    # Naive test at every look
    naive_rejections += naive_peeking(
        control, treatment, alpha=alpha
    )

msprt_fpr = msprt_rejections / N
naive_fpr = naive_rejections / N
mc_se = np.sqrt(alpha * (1 - alpha) / N)

print(f"Nominal alpha:              {alpha:.3f}")
print(f"mSPRT false positive rate:  {msprt_fpr:.3f} ({msprt_rejections}/{N})")
print(f"Naive z-test FPR:           {naive_fpr:.3f} ({naive_rejections}/{N})")
print(f"Monte Carlo SE at alpha:    {mc_se:.4f}")

assert 0 <= msprt_fpr <= 1
assert 0 <= naive_fpr <= 1


Nominal alpha:              0.050
mSPRT false positive rate:  0.030 (30/1000)
Naive z-test FPR:           0.702 (702/1000)
Monte Carlo SE at alpha:    0.0069


*Your answer here.*

I used Claude to help code and test an mSPRT. I compared it to a regular z-test across 1,000 null simulations. The mSPRT had a false positive rate of 0.030, while the regular z-test had a rate of 0.702. This showed how repeatedly testing can lead to way more false positives.

### Task 4.2: The CFO memo

*Your answer here.*

The current rule with p < 0.05 gave a false positive rate of 0.182 and a realized return of 0.275 which was better than the alternative with cohends d. at 200 experiments per year, the clients rule out porfomed the alternative and I would recomend sticking with that.  

### AI Transcript

Paste **both prompt versions exactly as you sent them** and the model's replies **exactly as they came back**, then one line on each change you made. Name the model and the date. For 5200 the transcript lives in `ai-appendix.md` in your repository; link it here.

*Model and date:* Claude 5 Sonnet 10/17/26

*Version 1 prompt:*

```
[Prep] Act as an experimentation platform engineer who has implemented
sequential testing in production.

[Request] I have a simulated portfolio of 200 A/B tests with known true
effects. Implement an always-valid sequential test -- a mixture
sequential probability ratio test or an alpha-spending boundary -- that
allows continuous monitoring without inflating the Type I error rate.
Give me a function that takes a stream of observations and returns the
stopping decision at each look.

[Iterate] numpy and scipy only. Reuse my log structure. The function
must be testable against a pure-null stream.

[Mechanism Check] Explain why the naive approach of testing at every
look inflates Type I error, and precisely what the always-valid
construction gives up in exchange for continuous monitoring.

[Evaluate] Run it on 1,000 pure-null streams and report the empirical
false positive rate. If it is not close to 5%, tell me what is wrong.
```

*Reply:*

I’ll build it, run it, and check the numbers. One note first: your message mentions “my log structure” and the 200-test portfolio, but nothing was attached. I’ll use a plain per-look log (dict of arrays) that’s easy to remap, and simulate a portfolio.

The implementation is in msprt.py. The msprt_stream(x_ctrl, x_trt, alpha, tau2, sigma2) function takes paired observation streams and returns a per-look log with n, diff, log_lambda, p_always_valid, ci_lo, ci_hi, reject and stop_at. It uses numpy and scipy only. It is a two-sample mSPRT with a N(0, τ²) mixture prior on the effect. It rejects at the first look where Λₙ ≥ 1/α, and p_always_valid is the running minimum of 1/Λ. Since I didn’t have your log schema, rename the dict keys to match it.

Null results (1,000 streams × 5,000 looks, α = 0.05)
Method	Empirical FPR
mSPRT, known σ²	3.0%
mSPRT, plug-in σ², starting at n=30	3.5%
mSPRT, plug-in σ², starting at n=2	16.2% (broken)
Naive z-test at every look	70.2%

The 3.0% and 3.5% are below 5% by design, not by error. Ville’s inequality guarantees P(sup Λₙ ≥ 1/α) ≤ α. That is an upper bound, and it is loose here for two reasons. The test is not exact because Λₙ drifts below the boundary instead of hitting it exactly, and I stopped at 5,000 looks, not infinity. Anything at or under 5% is correct. Getting close to 5% would require a tighter boundary or a longer horizon.

The 16.2% is a real bug, not noise, and it is the thing to watch. With a plug-in variance and no burn-in, the sample variance at n = 2 to 5 is often tiny. That makes V tiny, which inflates Λ, so streams get rejected on early flukes. The Ville guarantee assumes the likelihood ratio is a martingale under H₀, and a data-dependent variance estimate at very small n breaks that. The fixes, in order of preference:

Use a known or historically estimated σ² (3.0%).
Enforce a burn-in so the variance estimate stabilizes (3.5% at n = 30).
Use a t-based or variance-robust mixture. That is more work and still approximate.
200-test portfolio (70% true nulls, the rest with effects drawn from N(0, 0.15²))

There were 0 false positives among 144 nulls and 34 of 56 true effects detected, with a median stopping time of 736 looks. The remaining 22 effects are mostly tiny and would need more than 5,000 looks.

Why naive peeking inflates Type I error

A fixed-sample test controls P(|Zₙ| > 1.96) at one pre-specified n. Continuous monitoring tests the event {∃ n: |Zₙ| > 1.96}. Zₙ is a standardized random walk, so by the law of the iterated logarithm, |Zₙ| exceeds any fixed constant infinitely often as n → ∞. The rejection probability therefore goes to 1 as the number of looks grows. The simulation shows this: 70% at 5,000 looks.

What the always-valid construction gives up
Power, or equivalently sample size. The boundary must hold at every n at once, so it is wider than the fixed-sample one. Λₙ ≥ 1/α is roughly a z-boundary growing like √(log n) instead of a constant 1.96. You need more data to detect the same effect, in exchange for validity at any stopping rule.
Sensitivity to τ². The mixture variance sets where the test is most powerful. A τ² far from the typical true effect hurts power. I used τ² = 0.25 for the null runs and 0.05 for the portfolio. In production, set τ² from historical effect sizes.
Exactness. The type I error is bounded by α but typically below it, as seen above.
Assumptions. The test needs i.i.d. normal-ish data with a known or well-behaved variance, as the 16.2% result shows.

If you share your actual log format or the real portfolio, I can wire it in and compare mSPRT against an alpha-spending boundary on the same streams

*Model and date:* Claude 5 Sonnet 10/17/26

*Version 2 prompt:*

```
[Prep] Act as an experimentation platform engineer who has implemented
sequential testing in production in python.

[Request] I have a simulated portfolio of 200 A/B tests with known true
effects. Implement an always-valid sequential test -- a mixture
sequential probability ratio test or an alpha-spending boundary -- that
allows continuous monitoring without inflating the Type I error rate.
Give me a function that takes a stream of observations and returns the
stopping decision at each look.

[Iterate] numpy and scipy only. Reuse my log structure. The function
must be testable against a pure-null stream.

[Mechanism Check] Explain why the naive approach of testing at every
look inflates Type I error, and precisely what the always-valid
construction gives up in exchange for continuous monitoring.Use a fixed random seed, run 1,000 null streams with 5,000 looks each at alpha=0.05, and report the observed false positive rate alongside the nominal rate.

[Evaluate] Run it on 1,000 pure-null streams and report the empirical
false positive rate. If it is not close to 5%, tell me what is wrong.
```

*Reply:*

Results (seed 20260427, 1,000 null streams × 5,000 looks, α = 0.05)
Method	False positive rate
Nominal	5.0%
mSPRT (always-valid)	4.0% (40/1000)
Naive z-test at every look	62.0% (620/1000)

The Monte Carlo standard error at 5% is about 0.7 points, so 4.0% is about 1.4 SE below nominal. That is consistent with a valid test and is not a bug. More on this below.

What I built (msprt.py)

msprt(stream, alpha, sigma_d, tau) takes a stream of paired differences (or an (n, 2) array of control and treatment) and returns a per-look log with these columns:

look
mean_diff
log_lambda
p_always_valid
stop

stop is monotone: once the boundary is crossed it stays True. It uses numpy only, plus scipy for the naive baseline. The null test is built in: run python msprt.py. first_stop(log) gives the stopping look.

The statistic is the Gaussian-mixture likelihood ratio with a N(0, τ²) prior on the effect:

Λₙ = √(σ²/(σ²+nτ²)) · exp( n²τ²·x̄² / (2σ²(σ²+nτ²)) )

It rejects when Λₙ ≥ 1/α.

Two things to flag:

Log structure: you didn’t paste your log format, so I invented one (the columns above). Send me yours and I’ll map the output onto it.
Portfolio: your 200-test portfolio wasn’t attached either, so I simulated one. Half the tests are null and half have true effects of 0.03–0.12 SD. On it, mSPRT had 3 false positives among 93 nulls, 71% power, and a median stopping look of about 1,022 among detected effects. Swap in your real data and the function works unchanged.
Why the naive approach inflates Type I error

A fixed-horizon test controls the probability that |Zₙ| > 1.96 at one pre-chosen n. Peeking asks a different question: does the path |Zₙ| cross 1.96 at any n? Under the null, Zₙ is a random walk scaled by 1/√n. By the law of the iterated logarithm, it exceeds √(2 log log n) infinitely often, which grows without bound but only very slowly. So the crossing probability keeps accumulating as looks are added, and in the limit it reaches 1. Over 5,000 looks it is already 62%. The looks are highly correlated, so the inflation is far less than 5,000 × 5%, but it is still severe.

What the always-valid construction gives up
Power and sample size. Under H1, Λₙ grows like exp(n·θ²/2σ²) while the mixture penalty is only about ½·log n. Even so, you need more data than a fixed-horizon test powered for the same effect, because the error budget is spread over every possible stopping time. This is the price of validity at every n.
Conservativeness. Ville’s inequality bounds the probability of ever crossing by α. It does not make it equal to α. A finite horizon, here 5,000 looks, cuts the crossing probability further. That is why you see 4% and not 5%.
A tuning parameter (τ). The mixture is most efficient near effects of size ~τ. I used τ = 0.5σ. A smaller τ gives better power for tiny effects at a late stop, and a larger τ gives faster detection of big effects. τ must be fixed before seeing data.
Known variance. I assumed σ_d is known. Plugging in a running estimate technically breaks the exact martingale guarantee, though with large n the effect is small. A t-mixture (Lindon et al.) fixes this properly. I can add it if you want.
Point estimates. The always-valid p-value and confidence sequence stay valid at any stopping time. The naive sample mean at stopping is still biased away from zero, because you stopped on a favorable excursion.
On your “if not close to 5%” check

4.0% against a nominal 5% is the expected behavior and not a defect. The guarantee is “≤ 5%”, and for it to be wrong you would see something above 5% by more than noise. If I had seen 8% or more, the likely causes would be a wrong σ_d (an underestimated variance inflates every Λₙ), a mixture-formula typo, or non-i.i.d. data. I could run a 20,000-stream version to tighten the estimate.

## Truth and headline numbers

Required, and it must stay the **last code cell**. Print every parameter you planted (the prior on effects and its parameters, the sample-size range, the seed, the ship rule), each beside the name of the cell that set it; then every number that appears in the memo, each beside the name of the cell that produced it. A number in the memo that this cell does not print is treated as unsupported.

In [ ]:
print('Truth and headline numbers')

# Truth and headline numbers
# This must be the LAST code cell in the notebook.

print("PARAMETERS")
print("Prior share of real effects: 0.25 [set in Step 1.1 simulation cell]")
print("Real effects distribution: Normal(mean=0.30, sd=0.10) [set in Step 1.1 simulation cell]")
print("Null effects: exactly 0 [set in Step 1.1 simulation cell]")
print("Sample-size range: 40-399 [set in Step 1.1 simulation cell]")
print("Outcome standard deviation: 1.0 [set in Step 1.1 simulation cell]")
print("Random seed: 3 [set in Step 1.1 simulation cell]")
print("Portfolio size: 200 [set in Step 1.1 simulation cell]")
print("Client ship rule: p < 0.05 [set in Step 1.2 evaluation cell]")
print("Alternative ship rule: Cohen's d > 0.2 [set in Step 3.1 comparison cell]")

print("\nCLIENT RULE RESULTS ")
client_results = ep.evaluate(log, "p")
for name, value in client_results.items():
    print(f"{name}: {value} [produced by Step 1.2 evaluation cell]")

print("\nALTERNATIVE RULE RESULTS")
alternative_results = ep.evaluate(log, "cohens_d")
for name, value in alternative_results.items():
    print(f"{name}: {value} [produced by Step 3.1 comparison cell]")

print("\n\SEQUENTIAL NULL CHECK")
print("Nominal alpha: 0.05 [set in sequential null-check cell]")
print("mSPRT FPR: 0.030 (30/1000) [produced by sequential null-check cell]")
print("Naive z-test FPR: 0.702 (702/1000) [produced by sequential null-check cell]")
print("Monte Carlo SE: 0.0069 [produced by sequential null-check cell]")


Truth and headline numbers
PARAMETERS
Prior share of real effects: 0.25 [set in Step 1.1 simulation cell]
Real effects distribution: Normal(mean=0.30, sd=0.10) [set in Step 1.1 simulation cell]
Null effects: exactly 0 [set in Step 1.1 simulation cell]
Sample-size range: 40-399 [set in Step 1.1 simulation cell]
Outcome standard deviation: 1.0 [set in Step 1.1 simulation cell]
Random seed: 3 [set in Step 1.1 simulation cell]
Portfolio size: 200 [set in Step 1.1 simulation cell]
Client ship rule: p < 0.05 [set in Step 1.2 evaluation cell]
Alternative ship rule: Cohen's d > 0.2 [set in Step 3.1 comparison cell]

CLIENT RULE RESULTS 
Realized Return: 12.119199279650719 [produced by Step 1.2 evaluation cell]
Number Shipped: 44 [produced by Step 1.2 evaluation cell]
False-Ship Rate: 0.18181818181818182 [produced by Step 1.2 evaluation cell]
Average True Effect: 0.27543634726478905 [produced by Step 1.2 evaluation cell]
Average Estimated Effect: 0.30651996236794615 [produced by Step 1.2 evalua

<>:27: SyntaxWarning: invalid escape sequence '\S'
<>:27: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_2245/3567004347.py:27: SyntaxWarning: invalid escape sequence '\S'
  print("\n\SEQUENTIAL NULL CHECK")


## Before you submit

- [ ] Every heading above is still here, in order, with work under it.
- [ ] Every number you report is produced by a cell above it, and every claim has a text cell.
- [ ] The verification checkpoint(s) are met before anything built on them.
- [ ] Phase 4: the AI transcript is the exchange, not a summary; both prompt versions are in it.
- [ ] Upload the `.ipynb`, your `ai-appendix.md` and your `src/` files (or a PDF export) on Canvas; a Colab or Drive link alone is not a submission.
- [ ] GitHub portfolio: *File › Save a copy in GitHub*, path `Problem Set 2/Econ_5200_PS2.ipynb`, with a short `README.md` in that folder.
- [ ] Run *Runtime › Restart and run all* once, top to bottom, before you hand in.